# 03 — Visualizations

Healthcare Imaging Equipment Utilization & Downtime Analysis

**Month 3 milestone (Mar 2023):** downtime trend by machine type, utilization heatmap by hospital site, and the top failure causes chart. Static figures are saved to `outputs/figures/` (PNG, for the report and for recreating in Tableau) and an interactive version is saved as HTML.

In [1]:
import pathlib
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid')
FIG_DIR = pathlib.Path('../outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

## a) Downtime trend over time per machine type

In [2]:
downtime_trend = pd.read_csv('../outputs/exports/downtime_trend_by_type_month.csv')
downtime_trend['month'] = pd.to_datetime(downtime_trend['month'])

fig, ax = plt.subplots(figsize=(9, 5))
for machine_type, grp in downtime_trend.groupby('machine_type'):
    grp = grp.sort_values('month')
    ax.plot(grp['month'], grp['downtime_hours'], marker='o', label=machine_type)
ax.set_title('Monthly Downtime Hours by Machine Type')
ax.set_xlabel('Month')
ax.set_ylabel('Downtime hours')
ax.legend(title='Machine type')
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIG_DIR / 'downtime_trend_by_type.png', dpi=150)
plt.show()

/tmp/ipykernel_839/2702706925.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## b) Utilization heatmap by hospital site

In [3]:
site_month = pd.read_csv('../outputs/exports/utilization_by_site_month.csv')
pivot = site_month.pivot(index='hospital_site', columns='month', values='daily_usage_hours')

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, cmap='YlOrRd', annot=False, linewidths=0.5, ax=ax, cbar_kws={'label': 'Total usage hours'})
ax.set_title('Monthly Imaging Usage Hours by Hospital Site')
ax.set_xlabel('Month')
ax.set_ylabel('Hospital site')
fig.tight_layout()
fig.savefig(FIG_DIR / 'utilization_heatmap_by_site.png', dpi=150)
plt.show()

/tmp/ipykernel_839/4266334455.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## c) Top failure causes

In [4]:
top_causes = pd.read_csv('../outputs/exports/top_downtime_causes.csv')

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=top_causes, x='total_downtime_hours', y='issue_type', hue='issue_type', palette='mako', legend=False, ax=ax)
ax.set_title('Top 3 Causes of Downtime (by total hours)')
ax.set_xlabel('Total downtime hours')
ax.set_ylabel('Issue type')
fig.tight_layout()
fig.savefig(FIG_DIR / 'top_failure_causes.png', dpi=150)
plt.show()

/tmp/ipykernel_839/1551498075.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interactive version (Plotly)

Saved as a standalone HTML file so it can be opened without a notebook environment.

In [5]:
combined = pd.read_csv('../outputs/exports/utilization_downtime_by_machine.csv')
fig_px = px.scatter(
    combined, x='age_years', y='total_downtime_hours', size='ticket_count',
    color='machine_type', hover_name='machine_id',
    title='Machine Age vs. Total Downtime Hours (bubble size = ticket count)',
    labels={'age_years': 'Age (years)', 'total_downtime_hours': 'Total downtime (hours)'}
)
fig_px.write_html(str(FIG_DIR / 'age_vs_downtime_interactive.html'), include_plotlyjs='cdn')
fig_px.show()